In [1]:
import pandas as pd
import unicodedata
import glob
import os


### Sumário de padrões para tratamento de dados:

- Data: DD/MM/YYYY

- Data hora: DD/MM/YYYY HH:MM:SS

- Binário: 0 ou 1

- Strings e colunas: Maiusculas e sem acento: 

- Espaços: Manter apenas um espaço no intervalo entre palavras

- Sexo: M e F

- Delimitador: Vírgula

- Armazenamento: Todos os arquivos em .csv


### Funções para padronização:

In [2]:
# Formato em 2016-04-29T18:38:08Z
def padronizar_data_hora(df, coluna):

  df[coluna] = pd.to_datetime(df[coluna])
  
  df[coluna] = df[coluna].dt.strftime('%d/%m/%Y %H:%M:%S')
  
  return df


In [3]:
#Formato em MM/DD/AA
def padronizar_data(df, coluna):

  df[coluna] = pd.to_datetime(df[coluna], format='%m/%d/%Y')
  
  df[coluna] = df[coluna].dt.strftime('%d/%m/%Y')
  
  return df

In [4]:
def padronizar_colunas(df):

    df.columns = df.columns.str.upper()
    
    return df

In [5]:
def converter_para_binario(df, coluna):
    mapeamento = {'Yes': 1, 'No': 0}
    df[coluna].replace(mapeamento, inplace=True)
    return df

In [6]:
def remover_acentos(df):
    for coluna in df.columns:
        if df[coluna].dtype == 'object':
            df[coluna] = df[coluna].astype(str).str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    return df


In [7]:
def padronizar_maiusculo(df):
    for coluna in df.columns:
        if df[coluna].dtype == 'object':
            df[coluna] = df[coluna].astype(str).str.upper()
    return df

### Tratamento de dados

In [12]:
df_med_raw = pd.read_csv("dados/raw/medical_appointments.csv")
df_med = df_med_raw


In [34]:

df_clima_raw = pd.read_csv('dados/raw/meteorologia2016.csv', sep=';')
df_clima = df_clima_raw


In [ ]:
df_med = padronizar_data_hora(df_med, 'ScheduledDay')
df_med = padronizar_data_hora(df_med, 'AppointmentDay')
df_med = padronizar_colunas(df_med)
df_med = converter_para_binario(df_med, 'NO-SHOW')
df_med = remover_acentos(df_med)
df_med = padronizar_maiusculo(df_med)

In [27]:
df_clima.head()

,DATA,HORA_UTC,PRECIPITACAO_MM,PRESSAO_ESTACAO_MB,PRESSAO_MAX_MB,PRESSAO_MIN_MB,RADIACAO_KJ_M2,TEMP_AR_C,TEMP_ORVALHO_C,TEMP_MAX_C,TEMP_MIN_C,TEMP_ORVALHO_MAX_C,TEMP_ORVALHO_MIN_C,UMIDADE_MAX,UMIDADE_MIN,UMIDADE_RELATIVA,VENTO_DIRECAO_GRAUS,VENTO_RAJADA_MAX_MS,VENTO_VELOCIDADE_MS
0,2016-01-01,00:00:00,0,"924,1","924,1","923,6",-9999,"23,8","19,1","25,1","23,8","19,2","18,8",75,69,75,322,7,"3,4"
1,2016-01-01,01:00:00,0,"924,4","924,5","924,1",-9999,23,"18,7","23,8",23,"19,2","18,7",77,75,77,325,7,"3,7"
2,2016-01-01,02:00:00,0,"924,1","924,6","924,1",-9999,"22,5","18,4","23,1","22,4","18,8","18,4",78,76,77,334,"7,8","3,3"
3,2016-01-01,03:00:00,0,"923,9","924,1","923,8",-9999,"22,2","18,2","22,5",22,"18,4","18,1",79,77,78,306,"7,3","2,2"
4,2016-01-01,04:00:00,0,"923,3",924,"923,3",-9999,"21,7",18,"22,2","21,7","18,2",18,79,78,79,322,"7,7","4,2"


In [ ]:


# 1. Renomear colunas e remover a desnecessária (código original)
df_clima.columns = [
    "DATA", "HORA_UTC", "PRECIPITACAO_MM", "PRESSAO_ESTACAO_MB", "PRESSAO_MAX_MB",
    "PRESSAO_MIN_MB", "RADIACAO_KJ_M2", "TEMP_AR_C", "TEMP_ORVALHO_C", "TEMP_MAX_C",
    "TEMP_MIN_C", "TEMP_ORVALHO_MAX_C", "TEMP_ORVALHO_MIN_C", "UMIDADE_MAX",
    "UMIDADE_MIN", "UMIDADE_RELATIVA", "VENTO_DIRECAO_GRAUS", "VENTO_RAJADA_MAX_MS",
    "VENTO_VELOCIDADE_MS", "DESCARTAR"
]
df_clima = df_clima.drop(columns=["DESCARTAR"])


# 2. **NOVO**: Combinar data e hora em uma única coluna datetime
# Esta é uma boa prática para trabalhar com séries temporais.
df_clima['DATETIME_UTC'] = pd.to_datetime (df_clima['DATA'] + ' ' + df_clima['HORA_UTC'], errors='coerce')


# 3. **NOVO**: Criar as colunas "data" e "hora" formatadas como solicitado
# Note que estamos criando novas colunas para não interferir com a lógica original que usa "DATA".
df_clima['data_formatada'] = df_clima['DATETIME_UTC'].dt.strftime('%d/%m/%Y')
df_clima['hora_formatada'] = df_clima['DATETIME_UTC'].dt.strftime('%H:%M:%S')


# 4. Converter tipos de dados (código original ajustado)
df_clima["DATA"] = pd.to_datetime(df_clima["DATA"], errors="coerce")

# Ajuste para não tentar converter as novas colunas de data/hora para número
num_cols = df_clima.columns.drop([
    "DATA", "HORA_UTC", "DATETIME_UTC", "data_formatada", "hora_formatada"
])
for col in num_cols:
    df_clima[col] = (
        df_clima[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace("-9999", "0")
        .astype(float)
    )

# Neste ponto, o DataFrame `df_clima` já contém os dados horários
# com as colunas 'data_formatada' e 'hora_formatada' que você pediu.
# Você pode usá-lo como sua saída final se precisar dos dados por hora.
# Ex: print(df_clima[['data_formatada', 'hora_formatada', 'TEMP_AR_C']].head())


# --- O restante do seu código para agregação diária permanece o mesmo ---
# Lembre-se que o df_diario, por ser um resumo do DIA, não terá a coluna de hora.

df_diario = df_clima.groupby("DATA").agg({
    "PRECIPITACAO_MM": "sum",
    "TEMP_AR_C": ["mean", "max", "min"],
    "UMIDADE_RELATIVA": "mean",
    "VENTO_VELOCIDADE_MS": "mean"
}).reset_index()


df_diario.columns = [
    "DATA", "PRECIPITACAO_TOTAL_MM", "TEMP_MEDIA_C", "TEMP_MAXIMA_C",
    "TEMP_MINIMA_C", "UMIDADE_MEDIA", "VENTO_VELOCIDADE_MEDIA_MS"
]


num_cols_diario = df_diario.columns.drop("DATA")
df_diario[num_cols_diario] = df_diario[num_cols_diario].round(2)


df_diario["DATA_ANO_ANTERIOR"] = df_diario["DATA"] - pd.DateOffset(years=1)

df_aux = df_diario[["DATA", "TEMP_MEDIA_C", "TEMP_MAXIMA_C", "TEMP_MINIMA_C", "UMIDADE_MEDIA", "VENTO_VELOCIDADE_MEDIA_MS"]].copy()
df_aux = df_aux.rename(columns={
    "DATA": "DATA_ANO_ANTERIOR",
    "TEMP_MEDIA_C": "TEMP_MEDIA_C_ANO_ANTERIOR",
    "TEMP_MAXIMA_C": "TEMP_MAXIMA_C_ANO_ANTERIOR",
    "TEMP_MINIMA_C": "TEMP_MINIMA_C_ANO_ANTERIOR",
    "UMIDADE_MEDIA": "UMIDADE_MEDIA_ANO_ANTERIOR",
    "VENTO_VELOCIDADE_MEDIA_MS": "VENTO_VELOCIDADE_MEDIA_MS_ANO_ANTERIOR"
})

df_diario = df_diario.merge(df_aux, on="DATA_ANO_ANTERIOR", how="left")

df_diario = df_diario.drop(columns=[
    "DATA_ANO_ANTERIOR",
    "TEMP_MEDIA_C_ANO_ANTERIOR",
    "TEMP_MAXIMA_C_ANO_ANTERIOR",
    "TEMP_MINIMA_C_ANO_ANTERIOR",
    "UMIDADE_MEDIA_ANO_ANTERIOR",
    "VENTO_VELOCIDADE_MEDIA_MS_ANO_ANTERIOR"
])


def definir_estacao(data):
    mes = data.month
    dia = data.day
    if (mes == 12 and dia >= 21) or (mes in [1, 2]) or (mes == 3 and dia < 21):
        return "VERAO"
    elif (mes == 3 and dia >= 21) or (mes in [4, 5]) or (mes == 6 and dia < 21):
        return "OUTONO"
    elif (mes == 6 and dia >= 21) or (mes in [7, 8]) or (mes == 9 and dia < 22):
        return "INVERNO"
    elif (mes == 9 and dia >= 22) or (mes in [10, 11]) or (mes == 12 and dia < 21):
        return "PRIMAVERA"
    else:
        return pd.NA
df_diario["CLASSIFICACAO_ESTACAO"] = df_diario["DATA"].apply(definir_estacao)


def classificar_temp_media(temp):
    if temp < 10:
        return "MUITO_FRIO"
    elif 10 <= temp < 17:
        return "FRIO"
    elif 17 <= temp < 24:
        return "AGRADAVEL"
    elif 24 <= temp < 30:
        return "QUENTE"
    elif temp >= 30:
        return "MUITO_QUENTE"
    else:
        return pd.NA

df_diario["CLASSIFICACAO_TEMP_MEDIA"] = df_diario["TEMP_MEDIA_C"].apply(classificar_temp_media)


dias_semana = {
    "Monday": "SEGUNDA-FEIRA", "Tuesday": "TERÇA-FEIRA", "Wednesday": "QUARTA-FEIRA",
    "Thursday": "QUINTA-FEIRA", "Friday": "SEXTA-FEIRA", "Saturday": "SÁBADO",
    "Sunday": "DOMINGO"
}
df_diario["DIA_SEMANA"] = df_diario["DATA"].dt.day_name().map(dias_semana)

df_diario["DATA"] = df_diario["DATA"].dt.strftime("%d/%m/%Y")

In [39]:
df_clima.head(17)

,DATA,HORA_UTC,PRECIPITACAO_MM,PRESSAO_ESTACAO_MB,PRESSAO_MAX_MB,PRESSAO_MIN_MB,RADIACAO_KJ_M2,TEMP_AR_C,TEMP_ORVALHO_C,TEMP_MAX_C,...,TEMP_ORVALHO_MIN_C,UMIDADE_MAX,UMIDADE_MIN,UMIDADE_RELATIVA,VENTO_DIRECAO_GRAUS,VENTO_RAJADA_MAX_MS,VENTO_VELOCIDADE_MS,DATETIME_UTC,data_formatada,hora_formatada
0,2016-01-01,00:00,0.0,924.1,924.1,923.6,0.0,23.8,19.1,25.1,...,18.8,75.0,69.0,75.0,322.0,7.0,3.4,2016-01-01 00:00:00,01/01/2016,00:00:00
1,2016-01-01,01:00,0.0,924.4,924.5,924.1,0.0,23.0,18.7,23.8,...,18.7,77.0,75.0,77.0,325.0,7.0,3.7,2016-01-01 01:00:00,01/01/2016,01:00:00
2,2016-01-01,02:00,0.0,924.1,924.6,924.1,0.0,22.5,18.4,23.1,...,18.4,78.0,76.0,77.0,334.0,7.8,3.3,2016-01-01 02:00:00,01/01/2016,02:00:00
3,2016-01-01,03:00,0.0,923.9,924.1,923.8,0.0,22.2,18.2,22.5,...,18.1,79.0,77.0,78.0,306.0,7.3,2.2,2016-01-01 03:00:00,01/01/2016,03:00:00
4,2016-01-01,04:00,0.0,923.3,924.0,923.3,0.0,21.7,18.0,22.2,...,18.0,79.0,78.0,79.0,322.0,7.7,4.2,2016-01-01 04:00:00,01/01/2016,04:00:00
5,2016-01-01,05:00,0.0,922.9,923.4,922.8,0.0,21.4,18.1,21.7,...,18.0,82.0,79.0,82.0,299.0,7.7,2.7,2016-01-01 05:00:00,01/01/2016,05:00:00
6,2016-01-01,06:00,0.0,922.7,923.0,922.6,0.0,21.5,18.1,21.5,...,18.1,82.0,81.0,81.0,315.0,6.0,2.8,2016-01-01 06:00:00,01/01/2016,06:00:00
7,2016-01-01,07:00,0.0,922.5,922.8,922.4,0.0,21.5,17.9,21.6,...,17.9,81.0,80.0,80.0,321.0,8.4,3.3,2016-01-01 07:00:00,01/01/2016,07:00:00
8,2016-01-01,08:00,0.0,922.8,922.8,922.4,0.0,21.5,17.8,21.5,...,17.8,81.0,80.0,80.0,319.0,7.0,2.2,2016-01-01 08:00:00,01/01/2016,08:00:00
9,2016-01-01,09:00,0.0,923.1,923.2,922.8,45.8,21.8,18.1,21.8,...,17.8,80.0,79.0,80.0,336.0,5.5,1.8,2016-01-01 09:00:00,01/01/2016,09:00:00


In [ ]:
df_med_trusted = df_med
df_med_trusted.to_csv('01. Dataset - Medical Appointment No Shows/trusted/medical_appointment_no_show.csv')